# Pre-entrega 7: API de producción y monitoreo activo
API FastAPI asíncrona que expone el orquestador multi-agente de la Pre-entrega 6, con estado en Redis, trazas en Arize Phoenix y aprobación humana (HITL) para acciones críticas.

## 1. Instalación de librerías

In [1]:
!pip install -q fastapi uvicorn httpx redis langgraph langgraph-checkpoint-redis langchain-core langchain-google-genai arize-phoenix arize-phoenix-otel openinference-instrumentation-langchain python-dotenv

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.2 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.3.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
ibis-framework 9.5.0 requires sqlglot<25.21,>=23.4, but you have sqlglot 30.21.0 which is incompatible.


## 2. Instalación e inicio de Redis Stack
Redis con los módulos JSON y Search, que necesita el checkpointer de LangGraph (RedisSaver).

In [3]:
%%bash
CODENAME=$(. /etc/os-release && echo $VERSION_CODENAME)
curl -fsSL https://packages.redis.io/gpg | gpg --dearmor --yes -o /usr/share/keyrings/redis-archive-keyring.gpg
echo "deb [signed-by=/usr/share/keyrings/redis-archive-keyring.gpg] https://packages.redis.io/deb $CODENAME main" > /etc/apt/sources.list.d/redis.list
apt-get update -qq > /dev/null 2>&1
apt-get install -y -qq redis > /dev/null 2>&1

echo "--- version ---"
redis-server --version

echo "--- modulos encontrados ---"
MODULOS=$(find / \( -name "redisearch.so" -o -name "rejson.so" \) 2>/dev/null)
echo "$MODULOS"

ARGS=""
for m in $MODULOS; do ARGS="$ARGS --loadmodule $m"; done
redis-server --daemonize yes $ARGS
sleep 3

echo "--- prueba ---"
redis-cli ping
redis-cli MODULE LIST | grep -iE "search|json"

--- version ---
Redis server v=8.10.3 sha=00000000:1 malloc=jemalloc-5.3.0 bits=64 build=d635cf9cedec3d62
--- modulos encontrados ---
/usr/lib/redis/modules/rejson.so
/usr/lib/redis/modules/redisearch.so
--- prueba ---
PONG
search
/usr/lib/redis/modules/redisearch.so
ReJSON
/usr/lib/redis/modules/rejson.so


## 3. Verificación de Redis desde Python

In [4]:
import redis

r = redis.Redis(host="localhost", port=6379, decode_responses=True)
print("Ping:", r.ping())
print("Módulos:", sorted(m["name"] for m in r.module_list()))

Ping: True
Módulos: ['ReJSON', 'search', 'vectorset']


## 4. Inicio de Arize Phoenix (plataforma de observabilidad)
Corre en segundo plano en el puerto 6006 y recibe las trazas de la API.

In [37]:
import subprocess
import time

from google.colab.output import eval_js

subprocess.Popen("setsid nohup phoenix serve > phoenix.log 2>&1 &", shell=True)
time.sleep(15)

URL_PHOENIX = eval_js("google.colab.kernel.proxyPort(6006)")
print("Phoenix:", URL_PHOENIX)

Phoenix: https://6006-m-s-use1b7-kd99wir3ntnp-b.us-east1.prod.r.colab.dev


## 5. Variables de entorno para la API
La API corre como un proceso aparte, así que lee la configuración de un archivo .env (que no se sube al repo).

In [6]:
import os
from google.colab import userdata

os.makedirs("app/agents", exist_ok=True)
os.makedirs("screenshots", exist_ok=True)

with open(".env", "w") as f:
    f.write(f"GOOGLE_API_KEY={userdata.get('GOOGLE_API_KEY')}\n")
    f.write("REDIS_URL=redis://localhost:6379\n")
    f.write("PHOENIX_COLLECTOR_ENDPOINT=http://localhost:6006/v1/traces\n")

print("Archivo .env creado. Carpetas:", sorted(os.listdir(".")))

Archivo .env creado. Carpetas: ['.config', '.env', 'app', 'phoenix.log', 'sample_data', 'screenshots']


## 6. Código del agente (se escribe en la carpeta app/)

In [7]:
%%writefile app/__init__.py
"""API de producción del orquestador multi-agente."""

Writing app/__init__.py


In [8]:
%%writefile app/config.py
"""Configuración compartida y utilidades."""
import logging
import os
import unicodedata

from dotenv import load_dotenv
from langchain_core.messages import BaseMessage
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()


def _lista(variable: str, por_defecto: str) -> list[str]:
    return [m.strip() for m in os.getenv(variable, por_defecto).split(",") if m.strip()]


REDIS_URL: str = os.getenv("REDIS_URL", "redis://localhost:6379")
PHOENIX_COLLECTOR_ENDPOINT: str = os.getenv("PHOENIX_COLLECTOR_ENDPOINT", "http://localhost:6006/v1/traces")
PHOENIX_PROJECT_NAME: str = os.getenv("PHOENIX_PROJECT_NAME", "orquestador-api")

# Cada lista es un modelo principal y sus respaldos (si uno falla, se usa el siguiente)
MODELOS_SUPERVISOR: list[str] = _lista(
    "MODELOS_SUPERVISOR", "gemini-3.1-flash-lite,gemini-3.5-flash-lite,gemini-flash-lite-latest")
MODELOS_ESPECIALISTAS: list[str] = _lista(
    "MODELOS_ESPECIALISTAS", "gemini-3.5-flash,gemini-3.1-flash-lite,gemini-3.5-flash-lite")

MAX_PASOS_SUPERVISOR: int = 6          # condición de parada del supervisor
RECURSION_LIMIT_ESPECIALISTA: int = 8
RECURSION_LIMIT_GRAFO: int = 25
PAUSA_ENTRE_NODOS: int = int(os.getenv("PAUSA_ENTRE_NODOS", "5"))   # segundos (plan gratuito)
WORKERS: int = int(os.getenv("WORKERS", "2"))                       # trabajos procesados en paralelo
JOB_TTL_SEGUNDOS: int = 60 * 60 * 24                                # los trabajos se guardan 24 horas


def configurar_logs() -> None:
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)-7s | %(name)s | %(message)s",
        datefmt="%H:%M:%S",
    )
    for ruidoso in ("httpx", "google_genai", "opentelemetry"):
        logging.getLogger(ruidoso).setLevel(logging.WARNING)


def crear_llm(modelo: str, reintentos: int = 1) -> ChatGoogleGenerativeAI:
    return ChatGoogleGenerativeAI(model=modelo, temperature=0, max_retries=reintentos)


def texto_de(mensaje: BaseMessage) -> str:
    """Extrae el texto de un mensaje (Gemini a veces devuelve el contenido en bloques)."""
    contenido = mensaje.content
    if isinstance(contenido, str):
        return contenido
    return "".join(b.get("text", "") for b in contenido if isinstance(b, dict) and b.get("type") == "text")


def normalizar(texto: str) -> str:
    """Minúsculas y sin tildes, para comparar textos."""
    sin_tildes = unicodedata.normalize("NFD", texto.lower())
    return "".join(c for c in sin_tildes if unicodedata.category(c) != "Mn")

Writing app/config.py


In [9]:
%%writefile app/datos.py
"""Datos simulados: catálogo de productos y reseñas de clientes."""
from typing import TypedDict


class Producto(TypedDict):
    producto_id: str
    nombre: str
    categoria: str
    precio: int
    bateria_horas: int
    cancelacion_ruido: bool
    garantia_meses: int


class Resena(TypedDict):
    producto_id: str
    puntaje: int
    texto: str


PRODUCTOS: list[Producto] = [
    {"producto_id": "P-101", "nombre": "Auriculares Pulse X1", "categoria": "auriculares",
     "precio": 45000, "bateria_horas": 30, "cancelacion_ruido": True, "garantia_meses": 12},
    {"producto_id": "P-102", "nombre": "Auriculares Nova Pro", "categoria": "auriculares",
     "precio": 62000, "bateria_horas": 40, "cancelacion_ruido": True, "garantia_meses": 18},
    {"producto_id": "P-103", "nombre": "Auriculares Basic 200", "categoria": "auriculares",
     "precio": 18000, "bateria_horas": 12, "cancelacion_ruido": False, "garantia_meses": 6},
]

RESENAS: list[Resena] = [
    {"producto_id": "P-101", "puntaje": 5, "texto": "Excelente sonido y muy cómodos, los uso todo el día."},
    {"producto_id": "P-101", "puntaje": 4, "texto": "Buena batería, aunque el estuche es un poco frágil."},
    {"producto_id": "P-101", "puntaje": 5, "texto": "Los recomiendo, la cancelación de ruido es genial."},
    {"producto_id": "P-101", "puntaje": 3, "texto": "Suenan bien pero se desconectan a veces del celular."},
    {"producto_id": "P-101", "puntaje": 4, "texto": "Muy buena relación precio calidad."},
    {"producto_id": "P-102", "puntaje": 5, "texto": "Sonido increíble y la batería dura muchísimo."},
    {"producto_id": "P-102", "puntaje": 2, "texto": "Muy caros para lo que ofrecen, el micrófono es malo."},
    {"producto_id": "P-102", "puntaje": 4, "texto": "Cómodos y con buena cancelación de ruido."},
    {"producto_id": "P-102", "puntaje": 3, "texto": "Buenos, pero se calientan las orejas después de un rato."},
    {"producto_id": "P-102", "puntaje": 5, "texto": "Perfectos para viajar, excelente calidad."},
    {"producto_id": "P-103", "puntaje": 3, "texto": "Cumplen, pero el sonido es débil."},
    {"producto_id": "P-103", "puntaje": 2, "texto": "Se rompió el cable al mes, mala calidad."},
    {"producto_id": "P-103", "puntaje": 4, "texto": "Para el precio están bien."},
]

Writing app/datos.py


In [10]:
%%writefile app/state.py
"""Esquema del estado compartido entre el supervisor, los especialistas y el nodo HITL."""
import operator
from typing import Annotated, Any, TypedDict

from langgraph.graph import MessagesState


class Contribucion(TypedDict):
    paso: int
    agente: str
    instruccion: str
    herramientas_usadas: list[str]
    resultado: str


class EstadoOrquestador(MessagesState):
    """Estado compartido. Registra qué agente aportó qué información."""
    siguiente: str                                               # a quién delega el supervisor
    instruccion: str                                             # tarea puntual para el especialista
    contribuciones: Annotated[list[Contribucion], operator.add]  # se acumulan, no se pisan
    pasos_supervisor: int                                        # contador para la condición de parada
    tarea_completa: bool
    requiere_aprobacion: bool                                    # la solicitud incluye una acción crítica
    aprobacion: dict[str, Any]                                   # decisión del humano
    accion_ejecutada: str                                        # resultado de la acción crítica
    respuesta_final: str


def resumir_contribuciones(contribuciones: list[Contribucion]) -> str:
    """Texto con las contribuciones, para pasarle al supervisor y a la síntesis."""
    if not contribuciones:
        return "(todavía no hay contribuciones)"
    return "\n\n".join(
        f"[Paso {c['paso']}] {c['agente'].upper()} "
        f"(herramientas: {', '.join(c['herramientas_usadas']) or 'ninguna'})\n"
        f"Instrucción: {c['instruccion']}\nResultado:\n{c['resultado']}"
        for c in contribuciones
    )

Writing app/state.py


In [11]:
%%writefile app/agents/__init__.py
"""Agentes del orquestador: investigador, analista y supervisor."""

Writing app/agents/__init__.py


In [12]:
%%writefile app/agents/base.py
"""Lógica común para ejecutar un especialista, con modelos de respaldo."""
import asyncio
import logging
from typing import Any

from langchain_core.messages import AIMessage, HumanMessage, ToolMessage

from app.config import PAUSA_ENTRE_NODOS, RECURSION_LIMIT_ESPECIALISTA, texto_de
from app.state import Contribucion, EstadoOrquestador

logger = logging.getLogger("orquestador")


async def ejecutar_especialista(nombre: str, agentes: list[tuple[str, Any]], entrada: str,
                                state: EstadoOrquestador) -> dict[str, Any]:
    """Corre un especialista probando sus modelos en orden y registra su contribución."""
    await asyncio.sleep(PAUSA_ENTRE_NODOS)
    logger.info("%s recibe: %s", nombre, state["instruccion"])

    resultado, herramientas = None, []
    for modelo, agente in agentes:
        try:
            salida = await agente.ainvoke(
                {"messages": [HumanMessage(entrada)]},
                {"recursion_limit": RECURSION_LIMIT_ESPECIALISTA, "run_name": f"{nombre} ({modelo})"},
            )
            resultado = texto_de(salida["messages"][-1])
            herramientas = [m.name for m in salida["messages"] if isinstance(m, ToolMessage)]
            break
        except Exception as e:
            logger.warning("%s con %s falló (%s); se prueba el siguiente modelo",
                           nombre, modelo, type(e).__name__)

    if resultado is None:
        resultado = f"ERROR: el {nombre} no pudo completar la tarea con ningún modelo."

    logger.info("%s terminó usando %s", nombre, herramientas)
    contribucion: Contribucion = {
        "paso": len(state.get("contribuciones", [])) + 1,
        "agente": nombre,
        "instruccion": state["instruccion"],
        "herramientas_usadas": herramientas,
        "resultado": resultado,
    }
    return {"messages": [AIMessage(content=resultado, name=nombre)],
            "contribuciones": [contribucion]}

Writing app/agents/base.py


In [13]:
%%writefile app/agents/research_agent.py
"""Agente Investigador: busca productos y reseñas en la base de datos."""
import asyncio
import logging
from typing import Any, Callable

from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

from app.agents.base import ejecutar_especialista
from app.config import crear_llm, normalizar
from app.datos import PRODUCTOS, RESENAS
from app.state import EstadoOrquestador

logger = logging.getLogger("orquestador")

PROMPT_INVESTIGADOR = """Sos el agente Investigador de una tienda de tecnología.
Tu trabajo es buscar información en la base de datos con tus herramientas: productos y reseñas.
Reglas:
- Primero buscá cada producto para obtener su producto_id y después pedí sus reseñas.
- No hagas cálculos ni opines: solo reportá los datos encontrados.
- Devolvé para cada producto: producto_id, nombre, precio, características y la lista completa
  de reseñas (puntaje y texto), en un formato claro.
- Si no encontrás algo, decilo explícitamente."""


@tool
async def buscar_productos(consulta: str) -> dict[str, Any]:
    """Busca productos del catálogo por nombre, modelo o categoría (por ejemplo "Pulse X1",
    "Nova" o "auriculares").

    Devuelve los productos que coinciden con su producto_id, nombre, precio en pesos y
    características técnicas (batería, cancelación de ruido, garantía). Usala primero para
    obtener el producto_id, que se necesita para pedir las reseñas.
    """
    await asyncio.sleep(0.2)  # latencia simulada
    palabras = [p for p in normalizar(consulta).split() if len(p) >= 2]
    encontrados = [p for p in PRODUCTOS
                   if any(w in normalizar(f"{p['nombre']} {p['categoria']}") for w in palabras)]
    logger.info("buscar_productos(%r) -> %d resultado(s)", consulta, len(encontrados))
    if not encontrados:
        return {"error": f"No hay productos que coincidan con '{consulta}'."}
    return {"productos": encontrados}


@tool
async def obtener_resenas(producto_id: str) -> dict[str, Any]:
    """Devuelve todas las reseñas de clientes de un producto a partir de su producto_id
    (por ejemplo "P-101"). Cada reseña tiene un puntaje de 1 a 5 y el texto del comentario.
    El producto_id se obtiene antes con buscar_productos.
    """
    await asyncio.sleep(0.2)
    resenas = [{"puntaje": r["puntaje"], "texto": r["texto"]}
               for r in RESENAS if r["producto_id"] == producto_id]
    logger.info("obtener_resenas(%r) -> %d reseña(s)", producto_id, len(resenas))
    if not resenas:
        return {"error": f"No hay reseñas para el producto '{producto_id}'."}
    return {"producto_id": producto_id, "resenas": resenas}


HERRAMIENTAS_INVESTIGADOR = [buscar_productos, obtener_resenas]


def crear_nodo_investigador(modelos: list[str]) -> Callable:
    agentes = [(m, create_react_agent(crear_llm(m), HERRAMIENTAS_INVESTIGADOR,
                                      prompt=PROMPT_INVESTIGADOR, name="investigador"))
               for m in modelos]

    async def nodo_investigador(state: EstadoOrquestador) -> dict[str, Any]:
        # Solo recibe la instrucción del supervisor, no todo el historial
        return await ejecutar_especialista("investigador", agentes, state["instruccion"], state)

    return nodo_investigador

Writing app/agents/research_agent.py


In [14]:
%%writefile app/agents/analyst_agent.py
"""Agente Analista: estadísticas, análisis de sentimiento y cálculos."""
import ast
import logging
import operator as op
import statistics
from typing import Any, Callable

from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

from app.agents.base import ejecutar_especialista
from app.config import crear_llm, normalizar
from app.state import EstadoOrquestador

logger = logging.getLogger("orquestador")

PROMPT_ANALISTA = """Sos el agente Analista de una tienda de tecnología.
Recibís datos ya obtenidos por el Investigador y los procesás con tus herramientas.
Reglas:
- Calculá el puntaje promedio de cada producto con calcular_estadisticas.
- Analizá el sentimiento de las reseñas de cada producto con analizar_sentimiento.
- Hacé cualquier cuenta (por ejemplo, precio / promedio) con calcular, nunca de memoria.
- Devolvé los resultados numéricos de cada producto y una conclusión breve basada en ellos.
- Si te faltan datos para algún cálculo, decí cuáles."""

PALABRAS_POSITIVAS = {"excelente", "buen", "buena", "buenos", "comodo", "comodos", "genial",
                      "recomiendo", "increible", "perfecto", "perfectos", "bien", "dura"}
PALABRAS_NEGATIVAS = {"fragil", "desconectan", "caro", "caros", "malo", "mala", "calientan",
                      "debil", "rompio", "falla"}

OPERADORES = {ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul, ast.Div: op.truediv,
              ast.Pow: op.pow, ast.USub: op.neg}


@tool
def calcular_estadisticas(valores: list[float]) -> dict[str, Any]:
    """Calcula estadísticas de una lista de números (por ejemplo, los puntajes de las reseñas
    de un producto): cantidad, promedio, mínimo, máximo y desvío estándar.
    Usala para obtener el puntaje promedio de un producto en lugar de calcularlo de memoria.
    """
    if not valores:
        return {"error": "La lista de valores está vacía."}
    resultado = {
        "cantidad": len(valores),
        "promedio": round(statistics.mean(valores), 2),
        "minimo": min(valores),
        "maximo": max(valores),
        "desvio": round(statistics.pstdev(valores), 2),
    }
    logger.info("calcular_estadisticas(%s) -> promedio %.2f", valores, resultado["promedio"])
    return resultado


@tool
def analizar_sentimiento(textos: list[str]) -> dict[str, Any]:
    """Analiza el sentimiento de una lista de textos (por ejemplo, las reseñas de un producto)
    con un léxico de palabras positivas y negativas.

    Devuelve, para cada texto, si es positivo, negativo o mixto, y un resumen con la cantidad
    de cada tipo y el porcentaje de reseñas positivas.
    """
    detalle = []
    for texto in textos:
        palabras = set(normalizar(texto).replace(",", " ").replace(".", " ").split())
        pos, neg = len(palabras & PALABRAS_POSITIVAS), len(palabras & PALABRAS_NEGATIVAS)
        etiqueta = "positivo" if pos > neg else "negativo" if neg > pos else "mixto"
        detalle.append({"texto": texto, "sentimiento": etiqueta})

    conteo = {e: sum(d["sentimiento"] == e for d in detalle) for e in ("positivo", "mixto", "negativo")}
    resumen = {**conteo,
               "porcentaje_positivo": round(100 * conteo["positivo"] / len(textos), 1) if textos else 0}
    logger.info("analizar_sentimiento(%d textos) -> %s", len(textos), resumen)
    return {"resumen": resumen, "detalle": detalle}


def _evaluar(nodo: ast.AST) -> float:
    if isinstance(nodo, ast.Constant) and isinstance(nodo.value, (int, float)):
        return nodo.value
    if isinstance(nodo, ast.BinOp) and type(nodo.op) in OPERADORES:
        return OPERADORES[type(nodo.op)](_evaluar(nodo.left), _evaluar(nodo.right))
    if isinstance(nodo, ast.UnaryOp) and type(nodo.op) in OPERADORES:
        return OPERADORES[type(nodo.op)](_evaluar(nodo.operand))
    raise ValueError("Expresión no permitida")


@tool
def calcular(expresion: str) -> dict[str, Any]:
    """Calculadora: resuelve una expresión aritmética con +, -, *, /, ** y paréntesis,
    por ejemplo "45000 / 4.2". Usala para cualquier cuenta, como el precio por punto de
    puntaje (precio / promedio) o diferencias porcentuales, en lugar de calcular de memoria.
    """
    try:
        resultado = round(_evaluar(ast.parse(expresion, mode="eval").body), 2)
    except Exception as e:
        return {"error": f"No se pudo calcular '{expresion}': {e}"}
    logger.info("calcular(%r) -> %s", expresion, resultado)
    return {"expresion": expresion, "resultado": resultado}


HERRAMIENTAS_ANALISTA = [calcular_estadisticas, analizar_sentimiento, calcular]


def crear_nodo_analista(modelos: list[str]) -> Callable:
    agentes = [(m, create_react_agent(crear_llm(m), HERRAMIENTAS_ANALISTA,
                                      prompt=PROMPT_ANALISTA, name="analista"))
               for m in modelos]

    async def nodo_analista(state: EstadoOrquestador) -> dict[str, Any]:
        # Solo recibe su instrucción y los datos del investigador, no todo el historial
        datos = [c["resultado"] for c in state.get("contribuciones", []) if c["agente"] == "investigador"]
        entrada = (state["instruccion"] + "\n\nDATOS DEL INVESTIGADOR:\n"
                   + (datos[-1] if datos else "(no hay datos)"))
        return await ejecutar_especialista("analista", agentes, entrada, state)

    return nodo_analista

Writing app/agents/analyst_agent.py


In [15]:
%%writefile app/agents/supervisor.py
"""Supervisor (router con rúbrica de validación) y nodo de síntesis final."""
import asyncio
import logging
from typing import Any, Callable, Literal

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from pydantic import BaseModel, Field

from app.config import MAX_PASOS_SUPERVISOR, PAUSA_ENTRE_NODOS, crear_llm, texto_de
from app.state import EstadoOrquestador, resumir_contribuciones

logger = logging.getLogger("orquestador")


class DecisionSupervisor(BaseModel):
    siguiente: Literal["investigador", "analista", "FINALIZAR"] = Field(
        description="Quién interviene ahora, o FINALIZAR si el análisis está completo.")
    instruccion: str = Field(
        description="Tarea concreta y autocontenida para el especialista. Vacía si es FINALIZAR.")
    motivo: str = Field(description="Por qué se toma esta decisión, en una oración.")


PROMPT_SUPERVISOR = """Sos el Supervisor de un equipo de agentes de una tienda de tecnología.
Dada la solicitud del usuario y las contribuciones de los especialistas, decidí quién debe
intervenir ahora o si es momento de finalizar.

Especialistas disponibles:
- investigador: busca productos y reseñas en la base de datos. No hace cálculos.
- analista: calcula promedios, analiza el sentimiento y hace cuentas sobre datos ya obtenidos.
  No puede buscar información: necesita que el investigador haya trabajado antes.

Rúbrica de validación (TODOS los puntos deben cumplirse para FINALIZAR):
1. El investigador obtuvo precio y reseñas de TODOS los productos que menciona el usuario.
2. El analista calculó con herramientas todo lo que pide el usuario (promedios, sentimiento,
   comparación de precio/calidad) para todos esos productos.
3. Ninguna contribución contiene ERROR ni dice que faltan datos.

Si un punto no se cumple, delegá en el especialista que corresponda con una instrucción
específica de corrección. No repitas una tarea que ya está bien hecha.
Si el usuario pide reservar o comprar, NO lo delegues: esa acción la gestiona otro paso del
sistema con aprobación humana. Vos solo te ocupás de la búsqueda y el análisis."""

PROMPT_SINTESIS = """Sos el responsable de redactar la respuesta final para el cliente.
Usá SOLO la información de las contribuciones de los especialistas y el estado de la acción
solicitada; no agregues datos nuevos. Respondé en español, de forma clara y breve: comparación
de los productos, opinión de los clientes, puntajes y una recomendación final justificada con
los números. Si hubo una acción crítica (por ejemplo, una reserva), informá si se ejecutó o no."""


def crear_nodo_supervisor(modelos: list[str]) -> Callable:
    """Si el primer modelo falla, se usan los siguientes como respaldo."""
    llm_decisor = crear_llm(modelos[0]).with_structured_output(DecisionSupervisor).with_fallbacks(
        [crear_llm(m).with_structured_output(DecisionSupervisor) for m in modelos[1:]])

    async def nodo_supervisor(state: EstadoOrquestador) -> dict[str, Any]:
        pasos = state.get("pasos_supervisor", 0)
        solicitud = texto_de(state["messages"][0])

        if pasos >= MAX_PASOS_SUPERVISOR:
            logger.warning("Supervisor: se alcanzó el máximo de %d pasos -> se fuerza el cierre", pasos)
            decision = DecisionSupervisor(siguiente="FINALIZAR", instruccion="",
                                          motivo="Se alcanzó el límite de pasos del supervisor")
        else:
            if pasos > 0:
                await asyncio.sleep(PAUSA_ENTRE_NODOS)
            decision = await llm_decisor.ainvoke([
                SystemMessage(PROMPT_SUPERVISOR),
                HumanMessage(f"SOLICITUD DEL USUARIO:\n{solicitud}\n\nCONTRIBUCIONES HASTA AHORA:\n"
                             f"{resumir_contribuciones(state.get('contribuciones', []))}"),
            ])

        logger.info("SUPERVISOR (paso %d) -> %s | %s", pasos + 1, decision.siguiente, decision.motivo)
        return {
            "siguiente": decision.siguiente,
            "instruccion": decision.instruccion,
            "pasos_supervisor": pasos + 1,
            "tarea_completa": decision.siguiente == "FINALIZAR",
            "messages": [AIMessage(content=f"Decisión: {decision.siguiente}. "
                                           f"Motivo: {decision.motivo.rstrip('.')}. "
                                           f"Instrucción: {decision.instruccion}", name="supervisor")],
        }

    return nodo_supervisor


def enrutar(state: EstadoOrquestador) -> Literal["investigador", "analista", "aprobacion"]:
    """Arista condicional: al finalizar el análisis, pasa por el control de aprobación."""
    return "aprobacion" if state["siguiente"] == "FINALIZAR" else state["siguiente"]


def crear_nodo_sintesis(modelos: list[str]) -> Callable:
    llm = crear_llm(modelos[0]).with_fallbacks([crear_llm(m) for m in modelos[1:]])

    async def nodo_sintesis(state: EstadoOrquestador) -> dict[str, Any]:
        await asyncio.sleep(PAUSA_ENTRE_NODOS)
        solicitud = texto_de(state["messages"][0])
        accion = state.get("accion_ejecutada") or "No se solicitó ninguna acción crítica."
        respuesta = await llm.ainvoke([
            SystemMessage(PROMPT_SINTESIS),
            HumanMessage(f"SOLICITUD DEL USUARIO:\n{solicitud}\n\n"
                         f"CONTRIBUCIONES:\n{resumir_contribuciones(state.get('contribuciones', []))}\n\n"
                         f"ESTADO DE LA ACCIÓN SOLICITADA:\n{accion}"),
        ])
        texto = texto_de(respuesta)
        logger.info("SINTESIS: respuesta final generada (%d caracteres)", len(texto))
        return {"respuesta_final": texto, "messages": [AIMessage(content=texto, name="sintesis")]}

    return nodo_sintesis

Writing app/agents/supervisor.py


In [16]:
%%writefile app/hitl.py
"""Human-in-the-loop: pausa el grafo antes de ejecutar una acción crítica (reservar un producto)."""
import json
import logging
from datetime import datetime, timezone
from typing import Any

from langchain_core.runnables import RunnableConfig
from langgraph.types import interrupt
from redis.asyncio import Redis

from app.config import REDIS_URL, normalizar, texto_de
from app.state import EstadoOrquestador

logger = logging.getLogger("orquestador")

# Pedidos con efectos secundarios: generan una reserva real en el sistema
PALABRAS_CRITICAS = ("reserv", "apart", "orden de compra", "pagar", "cobrar")

_redis: Redis | None = None


def _cliente_redis() -> Redis:
    global _redis
    if _redis is None:
        _redis = Redis.from_url(REDIS_URL, decode_responses=True)
    return _redis


def es_accion_critica(solicitud: str) -> bool:
    """El sistema identifica como crítica cualquier solicitud que implique reservar o pagar."""
    texto = normalizar(solicitud)
    return any(p in texto for p in PALABRAS_CRITICAS)


async def registrar_reserva(job_id: str, solicitud: str, aprobacion: dict[str, Any]) -> str:
    """Efecto secundario simulado: guarda la reserva en Redis."""
    reserva = {
        "job_id": job_id,
        "solicitud": solicitud,
        "aprobado_por": aprobacion.get("operador", "operador"),
        "comentario": aprobacion.get("comentario", ""),
        "fecha": datetime.now(timezone.utc).isoformat(),
    }
    numero = await _cliente_redis().rpush("reservas", json.dumps(reserva, ensure_ascii=False))
    return f"Reserva N° {numero} registrada (aprobada por {reserva['aprobado_por']})."


async def nodo_aprobacion(state: EstadoOrquestador, config: RunnableConfig) -> dict[str, Any]:
    """Si la solicitud es crítica, detiene el grafo hasta recibir la aprobación humana."""
    solicitud = texto_de(state["messages"][0])
    if not es_accion_critica(solicitud):
        logger.info("HITL: la solicitud no requiere aprobación")
        return {"requiere_aprobacion": False}

    logger.info("HITL: acción crítica detectada -> se pausa hasta recibir aprobación")
    contribuciones = state.get("contribuciones", [])
    decision = interrupt({
        "tipo": "aprobacion_requerida",
        "accion": "Registrar una reserva del producto recomendado",
        "solicitud": solicitud,
        "analisis": contribuciones[-1]["resultado"][:1500] if contribuciones else "",
    })

    # Esto se ejecuta recién cuando llega la respuesta del humano (Command(resume=...))
    aprobado = bool(decision.get("aprobado"))
    if aprobado:
        job_id = config["configurable"]["thread_id"]
        accion = await registrar_reserva(job_id, solicitud, decision)
    else:
        accion = f"La reserva NO se realizó: fue rechazada por el operador ({decision.get('comentario', '')})."
    logger.info("HITL: decisión recibida (aprobado=%s) -> %s", aprobado, accion)
    return {"requiere_aprobacion": True, "aprobacion": decision, "accion_ejecutada": accion}

Writing app/hitl.py


## 7. Verificación del código (sin LLM)

In [17]:
import importlib
import sys

sys.path.insert(0, ".")
for modulo in ["app.config", "app.state", "app.agents.base", "app.agents.research_agent",
               "app.agents.analyst_agent", "app.agents.supervisor", "app.hitl"]:
    importlib.import_module(modulo)
    print("OK", modulo)

from app.hitl import es_accion_critica

print()
print("¿Crítica? 'Compará el Pulse X1 y el Nova Pro'          ->",
      es_accion_critica("Compará el Pulse X1 y el Nova Pro"))
print("¿Crítica? 'Compará y reservame el que convenga más'   ->",
      es_accion_critica("Compará el Pulse X1 y el Nova Pro y reservame el que convenga más"))

OK app.config
OK app.state
OK app.agents.base
OK app.agents.research_agent
OK app.agents.analyst_agent
OK app.agents.supervisor
OK app.hitl

¿Crítica? 'Compará el Pulse X1 y el Nova Pro'          -> False
¿Crítica? 'Compará y reservame el que convenga más'   -> True


## 8. Grafo con checkpointer de Redis

In [18]:
%%writefile app/graph.py
"""Orquestador multi-agente de la Pre-entrega 6 + nodo HITL + checkpointer de Redis."""
import logging

from langgraph.checkpoint.base import BaseCheckpointSaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.state import CompiledStateGraph

from app.agents.analyst_agent import crear_nodo_analista
from app.agents.research_agent import crear_nodo_investigador
from app.agents.supervisor import crear_nodo_sintesis, crear_nodo_supervisor, enrutar
from app.config import MODELOS_ESPECIALISTAS, MODELOS_SUPERVISOR
from app.hitl import nodo_aprobacion
from app.state import EstadoOrquestador

logger = logging.getLogger("orquestador")


def construir_grafo(checkpointer: BaseCheckpointSaver) -> CompiledStateGraph:
    logger.info("Supervisor/síntesis: %s | Especialistas: %s", MODELOS_SUPERVISOR, MODELOS_ESPECIALISTAS)

    builder = StateGraph(EstadoOrquestador)
    builder.add_node("supervisor", crear_nodo_supervisor(MODELOS_SUPERVISOR))
    builder.add_node("investigador", crear_nodo_investigador(MODELOS_ESPECIALISTAS))
    builder.add_node("analista", crear_nodo_analista(MODELOS_ESPECIALISTAS))
    builder.add_node("aprobacion", nodo_aprobacion)      # Human-in-the-loop
    builder.add_node("sintesis", crear_nodo_sintesis(MODELOS_SUPERVISOR))

    builder.add_edge(START, "supervisor")
    builder.add_conditional_edges("supervisor", enrutar)  # -> investigador / analista / aprobacion
    builder.add_edge("investigador", "supervisor")
    builder.add_edge("analista", "supervisor")
    builder.add_edge("aprobacion", "sintesis")
    builder.add_edge("sintesis", END)

    # Con checkpointer, el estado de cada trabajo (thread_id = job_id) queda guardado en Redis,
    # y eso es lo que permite pausar con interrupt() y reanudar más tarde.
    return builder.compile(checkpointer=checkpointer)

Writing app/graph.py


## 9. Observabilidad con Arize Phoenix

In [23]:
%%writefile app/observability.py
"""Instrumentación: envía a Arize Phoenix una traza por trabajo, con cada nodo y llamada al LLM."""
import logging

from openinference.instrumentation.langchain import LangChainInstrumentor
from opentelemetry.trace import Tracer
from phoenix.otel import register

from app.config import PHOENIX_COLLECTOR_ENDPOINT, PHOENIX_PROJECT_NAME

logger = logging.getLogger("orquestador")


def iniciar_observabilidad() -> Tracer:
    tracer_provider = register(
        project_name=PHOENIX_PROJECT_NAME,
        endpoint=PHOENIX_COLLECTOR_ENDPOINT,
        batch=True,
        verbose=False,   # sin el cartel de bienvenida de Phoenix
    )
    # Instrumenta automáticamente LangChain y LangGraph: nodos, agentes, herramientas y LLM
    LangChainInstrumentor().instrument(tracer_provider=tracer_provider)
    logger.info("Observabilidad activa: proyecto '%s' en %s", PHOENIX_PROJECT_NAME, PHOENIX_COLLECTOR_ENDPOINT)
    return tracer_provider.get_tracer("orquestador-api")

Overwriting app/observability.py


## 10. Worker: procesa los trabajos en segundo plano y guarda su estado en Redis
Estados: PENDING -> RUNNING -> (WAITING_APPROVAL) -> DONE o FAILED.

In [20]:
%%writefile app/worker.py
"""Cola de trabajos en segundo plano. El estado de cada trabajo se guarda en Redis."""
import asyncio
import json
import logging
from datetime import datetime, timezone
from typing import Any

from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph.state import CompiledStateGraph
from langgraph.types import Command
from opentelemetry.trace import Tracer
from redis.asyncio import Redis

from app.config import JOB_TTL_SEGUNDOS, RECURSION_LIMIT_GRAFO

logger = logging.getLogger("orquestador")

PENDING, RUNNING, WAITING_APPROVAL, DONE, FAILED = "PENDING", "RUNNING", "WAITING_APPROVAL", "DONE", "FAILED"


def _ahora() -> str:
    return datetime.now(timezone.utc).isoformat()


class GestorTrabajos:
    def __init__(self, grafo: CompiledStateGraph, redis: Redis, tracer: Tracer) -> None:
        self.grafo = grafo
        self.redis = redis
        self.tracer = tracer
        self.cola: asyncio.Queue[tuple[str, str, dict[str, Any]]] = asyncio.Queue()
        self.tareas: list[asyncio.Task] = []

    # ---------- Estado de los trabajos en Redis ----------
    async def guardar(self, job_id: str, **campos: Any) -> None:
        datos = {k: v if isinstance(v, str) else json.dumps(v, ensure_ascii=False) for k, v in campos.items()}
        datos["actualizado"] = _ahora()
        clave = f"job:{job_id}"
        await self.redis.hset(clave, mapping=datos)
        await self.redis.expire(clave, JOB_TTL_SEGUNDOS)

    async def obtener(self, job_id: str) -> dict[str, Any] | None:
        datos = await self.redis.hgetall(f"job:{job_id}")
        if not datos:
            return None
        for campo in ("resultado", "aprobacion_pendiente"):
            if campo in datos:
                datos[campo] = json.loads(datos[campo])
        return datos

    # ---------- Cola ----------
    async def encolar(self, job_id: str, tipo: str, datos: dict[str, Any]) -> None:
        await self.cola.put((job_id, tipo, datos))

    async def iniciar(self, cantidad: int) -> None:
        self.tareas = [asyncio.create_task(self._bucle(i)) for i in range(cantidad)]
        logger.info("%d worker(s) iniciados", cantidad)

    async def detener(self) -> None:
        for tarea in self.tareas:
            tarea.cancel()
        await asyncio.gather(*self.tareas, return_exceptions=True)

    async def _bucle(self, numero: int) -> None:
        while True:
            job_id, tipo, datos = await self.cola.get()
            try:
                await self._procesar(job_id, tipo, datos, numero)
            finally:
                self.cola.task_done()

    # ---------- Ejecución ----------
    async def _procesar(self, job_id: str, tipo: str, datos: dict[str, Any], numero: int) -> None:
        logger.info("Worker %d toma el trabajo %s (%s)", numero, job_id, tipo)
        await self.guardar(job_id, estado=RUNNING)
        config = {"configurable": {"thread_id": job_id}, "recursion_limit": RECURSION_LIMIT_GRAFO,
                  "metadata": {"job_id": job_id}, "run_name": "orquestador"}

        if tipo == "nuevo":
            entrada: Any = {"messages": [HumanMessage(datos["consulta"])],
                            "contribuciones": [], "pasos_supervisor": 0}
        else:
            entrada = Command(resume=datos)   # respuesta del humano al interrupt()

        nombre_span = f"trabajo {job_id}" + (" (reanudación)" if tipo != "nuevo" else "")
        with self.tracer.start_as_current_span(nombre_span, attributes={
            "openinference.span.kind": "CHAIN", "job.id": job_id, "job.tipo": tipo,
            "input.value": json.dumps(datos, ensure_ascii=False),
        }):
            try:
                if "forzar_error" in datos.get("consulta", ""):
                    raise RuntimeError("Falla simulada para probar el estado FAILED")

                resultado = await self.grafo.ainvoke(entrada, config)
                interrupciones = resultado.get("__interrupt__") or []

                if interrupciones:
                    await self.guardar(job_id, estado=WAITING_APPROVAL,
                                       aprobacion_pendiente=[i.value for i in interrupciones])
                    logger.info("Trabajo %s pausado: espera aprobación humana", job_id)
                else:
                    await self.guardar(job_id, estado=DONE, resultado=self._resumen(resultado))
                    logger.info("Trabajo %s terminado", job_id)

            except Exception as e:
                # Si el agente falla en segundo plano, el estado pasa a FAILED (el cliente no queda esperando)
                logger.exception("Trabajo %s falló", job_id)
                await self.guardar(job_id, estado=FAILED, error=f"{type(e).__name__}: {e}")

    @staticmethod
    def _resumen(resultado: dict[str, Any]) -> dict[str, Any]:
        return {
            "respuesta_final": resultado.get("respuesta_final", ""),
            "flujo": [m.name for m in resultado.get("messages", []) if isinstance(m, AIMessage)],
            "pasos_supervisor": resultado.get("pasos_supervisor", 0),
            "requiere_aprobacion": resultado.get("requiere_aprobacion", False),
            "accion_ejecutada": resultado.get("accion_ejecutada", ""),
            "contribuciones": [{"paso": c["paso"], "agente": c["agente"],
                                "herramientas_usadas": c["herramientas_usadas"]}
                               for c in resultado.get("contribuciones", [])],
        }

Writing app/worker.py


## 11. API FastAPI

In [21]:
%%writefile app/main.py
"""API REST: recibe tareas, las encola y devuelve un job_id sin bloquear."""
import json
import uuid
from contextlib import asynccontextmanager
from typing import Any

from fastapi import FastAPI, HTTPException, Request, status
from langgraph.checkpoint.redis.aio import AsyncRedisSaver
from pydantic import BaseModel, Field
from redis.asyncio import Redis

from app.config import REDIS_URL, WORKERS, configurar_logs
from app.graph import construir_grafo
from app.observability import iniciar_observabilidad
from app.worker import PENDING, WAITING_APPROVAL, GestorTrabajos, _ahora


@asynccontextmanager
async def lifespan(app: FastAPI):
    configurar_logs()
    tracer = iniciar_observabilidad()
    redis = Redis.from_url(REDIS_URL, decode_responses=True)
    async with AsyncRedisSaver.from_conn_string(REDIS_URL) as checkpointer:
        await checkpointer.asetup()
        gestor = GestorTrabajos(construir_grafo(checkpointer), redis, tracer)
        await gestor.iniciar(WORKERS)
        app.state.gestor = gestor
        app.state.redis = redis
        yield
        await gestor.detener()
    await redis.aclose()


app = FastAPI(title="Orquestador multi-agente", version="1.0.0", lifespan=lifespan)


class TareaIn(BaseModel):
    consulta: str = Field(..., min_length=5, description="Pedido del cliente en lenguaje natural")


class AprobacionIn(BaseModel):
    aprobado: bool
    comentario: str = ""
    operador: str = "operador"


@app.post("/tasks", status_code=status.HTTP_202_ACCEPTED)
async def crear_tarea(tarea: TareaIn, request: Request) -> dict[str, str]:
    """Encola la tarea y devuelve el job_id de inmediato (no espera al agente)."""
    gestor: GestorTrabajos = request.app.state.gestor
    job_id = uuid.uuid4().hex[:12]
    await gestor.guardar(job_id, id=job_id, estado=PENDING, consulta=tarea.consulta, creado=_ahora())
    await gestor.encolar(job_id, "nuevo", {"consulta": tarea.consulta})
    return {"job_id": job_id, "estado": PENDING}


@app.get("/tasks/{job_id}")
async def consultar_tarea(job_id: str, request: Request) -> dict[str, Any]:
    """Devuelve el estado actual del trabajo (para hacer polling)."""
    trabajo = await request.app.state.gestor.obtener(job_id)
    if trabajo is None:
        raise HTTPException(status_code=404, detail="Trabajo no encontrado")
    return trabajo


@app.post("/tasks/{job_id}/approve", status_code=status.HTTP_202_ACCEPTED)
async def aprobar_tarea(job_id: str, aprobacion: AprobacionIn, request: Request) -> dict[str, str]:
    """Respuesta humana a una tarea pausada: la reanuda con la decisión tomada."""
    gestor: GestorTrabajos = request.app.state.gestor
    trabajo = await gestor.obtener(job_id)
    if trabajo is None:
        raise HTTPException(status_code=404, detail="Trabajo no encontrado")
    if trabajo["estado"] != WAITING_APPROVAL:
        raise HTTPException(status_code=409, detail=f"El trabajo no espera aprobación (estado {trabajo['estado']})")
    await gestor.guardar(job_id, estado=PENDING)
    await gestor.encolar(job_id, "reanudar", aprobacion.model_dump())
    return {"job_id": job_id, "estado": PENDING}


@app.get("/reservas")
async def listar_reservas(request: Request) -> list[dict[str, Any]]:
    """Reservas registradas por la acción crítica (solo las aprobadas)."""
    return [json.loads(r) for r in await request.app.state.redis.lrange("reservas", 0, -1)]


@app.get("/health")
async def health(request: Request) -> dict[str, Any]:
    return {"api": "ok", "redis": await request.app.state.redis.ping(),
            "en_cola": request.app.state.gestor.cola.qsize()}

Writing app/main.py


## 12. Levantar la API
Corre como proceso aparte en el puerto 8000. Los logs quedan en api.log.

In [38]:
import subprocess
import time

import httpx
from google.colab.output import eval_js

subprocess.Popen("setsid nohup uvicorn app.main:app --host 0.0.0.0 --port 8000 > api.log 2>&1 &", shell=True)
time.sleep(2)
subprocess.Popen("nohup uvicorn app.main:app --host 0.0.0.0 --port 8000 > api.log 2>&1 &", shell=True)
time.sleep(20)

API = "http://localhost:8000"
try:
    print("Health:", httpx.get(f"{API}/health", timeout=10).json())
except Exception as e:
    print("La API no respondió:", e)

print("\nSwagger (para probar a mano):", eval_js("google.colab.kernel.proxyPort(8000)").rstrip("/") + "/docs")
print("\n--- últimas líneas de api.log ---")
print(open("api.log").read()[-2500:])

La API no respondió: [Errno 111] Connection refused

Swagger (para probar a mano): https://8000-m-s-use1b7-kd99wir3ntnp-b.us-east1.prod.r.colab.dev/docs

--- últimas líneas de api.log ---



## 13. Cliente de prueba: crear tareas y consultar su estado (polling)

In [31]:
import asyncio
import json

import httpx

API = "http://localhost:8000"
ESTADOS_FINALES = {"DONE", "FAILED", "WAITING_APPROVAL"}


async def crear_tarea(cliente: httpx.AsyncClient, consulta: str) -> str:
    r = await cliente.post(f"{API}/tasks", json={"consulta": consulta})
    print(f"POST /tasks -> {r.status_code} {r.json()}")
    return r.json()["job_id"]


async def esperar(cliente: httpx.AsyncClient, job_id: str, maximo: int = 600) -> dict:
    """Consulta el estado cada 3 segundos hasta que el trabajo termina o se pausa.
    Si la conexión se corta, reintenta (uvicorn cierra conexiones inactivas a los 5 s)."""
    anterior = None
    for _ in range(maximo // 3):
        try:
            trabajo = (await cliente.get(f"{API}/tasks/{job_id}")).json()
        except httpx.TransportError:
            await asyncio.sleep(1)
            continue
        if trabajo["estado"] != anterior:
            print(f"  [{job_id}] estado: {trabajo['estado']}")
            anterior = trabajo["estado"]
        if trabajo["estado"] in ESTADOS_FINALES:
            return trabajo
        await asyncio.sleep(3)
    raise TimeoutError(f"El trabajo {job_id} no terminó en {maximo} segundos")

## 14. Prueba HITL: una solicitud crítica se pausa hasta recibir aprobación

In [26]:
async with httpx.AsyncClient(timeout=30) as cliente:
    job_hitl = await crear_tarea(cliente, "Compará los Pulse X1 y los Nova Pro: puntaje promedio, "
                                          "opinión de los clientes y precio/calidad. Reservame el que convenga más.")
    trabajo = await esperar(cliente, job_hitl)

print("\nEstado:", trabajo["estado"])
print("Aprobación pendiente:")
print(json.dumps(trabajo.get("aprobacion_pendiente"), ensure_ascii=False, indent=2)[:1200])

POST /tasks -> 202 {'job_id': '62aea3ec2ba3', 'estado': 'PENDING'}
  [62aea3ec2ba3] estado: RUNNING
  [62aea3ec2ba3] estado: WAITING_APPROVAL

Estado: WAITING_APPROVAL
Aprobación pendiente:
[
  {
    "tipo": "aprobacion_requerida",
    "accion": "Registrar una reserva del producto recomendado",
    "solicitud": "Compará los Pulse X1 y los Nova Pro: puntaje promedio, opinión de los clientes y precio/calidad. Reservame el que convenga más.",
    "analisis": "Aquí tienes los resultados procesados del análisis estadístico, de sentimiento y de relación precio/calidad para ambos productos:\n\n---\n\n### 1. Auriculares Pulse X1\n* **Puntaje promedio:** 4.2 / 5 (Mínimo: 3, Máximo: 5, Desvío estándar: 0.75)\n* **Análisis de sentimiento:**\n  * **Positivos:** 3 (60%)\n  * **Mixtos:** 2 (40%)\n  * **Negativos:** 0 (0%)\n* **Precio:** $45.000\n* **Relación precio / puntaje promedio:** **$10.714,29 por punto**\n\n---\n\n### 2. Auriculares Nova Pro\n* **Puntaje promedio:** 3.8 / 5 (Mínimo: 2, Máximo

## 15. Aprobación humana: se reanuda el grafo

In [27]:
async with httpx.AsyncClient(timeout=30) as cliente:
    r = await cliente.post(f"{API}/tasks/{job_hitl}/approve",
                           json={"aprobado": True, "comentario": "Hay stock, adelante", "operador": "Guido"})
    print(f"POST /tasks/{job_hitl}/approve -> {r.status_code} {r.json()}")
    trabajo = await esperar(cliente, job_hitl)
    reservas = (await cliente.get(f"{API}/reservas")).json()

print("\nEstado:", trabajo["estado"])
print("Flujo:", " -> ".join(trabajo["resultado"]["flujo"]))
print("Acción ejecutada:", trabajo["resultado"]["accion_ejecutada"])
print("\nRespuesta final:\n", trabajo["resultado"]["respuesta_final"])
print("\nReservas en Redis:", json.dumps(reservas, ensure_ascii=False, indent=2))

POST /tasks/62aea3ec2ba3/approve -> 202 {'job_id': '62aea3ec2ba3', 'estado': 'PENDING'}
  [62aea3ec2ba3] estado: RUNNING
  [62aea3ec2ba3] estado: DONE

Estado: DONE
Flujo: supervisor -> investigador -> supervisor -> analista -> supervisor -> sintesis
Acción ejecutada: Reserva N° 1 registrada (aprobada por Guido).

Respuesta final:
 Aquí tenés la comparación solicitada entre los auriculares:

*   **Puntaje promedio:** Los **Pulse X1** tienen un promedio de **4.2/5**, mientras que los **Nova Pro** alcanzan **3.8/5**.
*   **Opinión de los clientes:** Los Pulse X1 cuentan con un 60% de opiniones positivas y ninguna negativa, destacándose por su comodidad y cancelación de ruido. Los Nova Pro, aunque tienen un 60% de opiniones positivas, presentan un 20% de reseñas negativas debido a problemas con el micrófono y la relación precio-producto.
*   **Relación precio/calidad:** Los Pulse X1 son más convenientes, con un costo de **$10.714,29 por punto** de satisfacción, frente a los **$16.315,79**

## 16. Prueba de falla: el estado pasa a FAILED
Si el agente falla en segundo plano, el trabajo queda en FAILED con el error, y el cliente no se queda esperando.

In [28]:
async with httpx.AsyncClient(timeout=30) as cliente:
    job_falla = await crear_tarea(cliente, "forzar_error: consulta de prueba")
    trabajo = await esperar(cliente, job_falla)

print("\nEstado:", trabajo["estado"])
print("Error:", trabajo.get("error"))

POST /tasks -> 202 {'job_id': 'e475ebd8b7a1', 'estado': 'PENDING'}
  [e475ebd8b7a1] estado: FAILED

Estado: FAILED
Error: RuntimeError: Falla simulada para probar el estado FAILED


## 17. Prueba de carga: 5 peticiones concurrentes
Las 5 tareas se envían al mismo tiempo. La API responde enseguida con un job_id (no se bloquea) y los workers las procesan en segundo plano.

In [33]:
import statistics
import time

CONSULTAS_CARGA = [
    "Compará los Pulse X1 y los Nova Pro: puntaje promedio, opinión de los clientes y precio/calidad.",
    "Compará los Basic 200 y los Pulse X1: puntaje promedio, opinión de los clientes y cuál conviene.",
    "Compará los Nova Pro y los Basic 200: puntaje promedio y relación precio/calidad.",
    "¿Qué opinan los clientes de los Basic 200? Quiero el puntaje promedio y el análisis de sentimiento.",
    "Compará los tres modelos de auriculares (Pulse X1, Nova Pro y Basic 200) por precio/calidad.",
]


async def enviar_y_esperar(cliente: httpx.AsyncClient, consulta: str) -> dict:
    inicio = time.perf_counter()
    r = await cliente.post(f"{API}/tasks", json={"consulta": consulta})
    tiempo_post = time.perf_counter() - inicio
    job_id = r.json()["job_id"]
    trabajo = await esperar(cliente, job_id, maximo=1800)
    return {"job_id": job_id, "estado": trabajo["estado"], "post_ms": round(tiempo_post * 1000),
            "total_s": round(time.perf_counter() - inicio, 1),
            "flujo": " -> ".join(trabajo.get("resultado", {}).get("flujo", [])),
            "error": trabajo.get("error", "")}


inicio_carga = time.perf_counter()
async with httpx.AsyncClient(timeout=30) as cliente:
    resultados_carga = await asyncio.gather(*(enviar_y_esperar(cliente, c) for c in CONSULTAS_CARGA))
duracion = time.perf_counter() - inicio_carga

print("\n" + "=" * 90)
print(f"{'job_id':<14}{'estado':<10}{'POST (ms)':>10}{'total (s)':>11}   flujo")
print("=" * 90)
for x in resultados_carga:
    print(f"{x['job_id']:<14}{x['estado']:<10}{x['post_ms']:>10}{x['total_s']:>11}   {x['flujo'] or x['error']}")

tiempos = sorted(x["total_s"] for x in resultados_carga)
print(f"\nDuración total de la prueba: {duracion:.1f} s")
print(f"Respuesta de POST /tasks: máximo {max(x['post_ms'] for x in resultados_carga)} ms (no bloquea)")
print(f"Latencia de punta a punta (cliente): p50 = {statistics.median(tiempos):.1f} s | "
      f"p95 = {statistics.quantiles(tiempos, n=20, method='inclusive')[18]:.1f} s")
print(f"Completados: {sum(x['estado'] == 'DONE' for x in resultados_carga)} de {len(resultados_carga)}")

  [9e26914bc2ee] estado: RUNNING
  [8c570d4b5f75] estado: RUNNING
  [ccd4a319a620] estado: PENDING
  [4b38388a47ba] estado: PENDING
  [eb96b8c604ff] estado: PENDING
  [9e26914bc2ee] estado: DONE
  [4b38388a47ba] estado: RUNNING


CancelledError: 

## 18. Seguimiento de los 5 trabajos de la prueba de carga
La celda anterior envió las 5 peticiones concurrentes; la API las aceptó y las encoló. El seguimiento desde el notebook se cortó con un ReadError (uvicorn cierra conexiones inactivas a los 5 segundos y se consultaba justo cada 5). La API no se vio afectada: esta celda retoma el seguimiento de los mismos 5 trabajos con un polling que reintenta ante cortes de conexión.

In [39]:
import statistics
from datetime import datetime

JOBS_CARGA = ["782d8d56dae2", "d688ad1f6df4", "b63102152463", "2ddca761231c", "ff86a014ee67"]

async with httpx.AsyncClient(timeout=30) as cliente:
    trabajos = await asyncio.gather(*(esperar(cliente, j, maximo=1800) for j in JOBS_CARGA))


def segundos(t: dict) -> float:
    return (datetime.fromisoformat(t["actualizado"]) - datetime.fromisoformat(t["creado"])).total_seconds()


print("\n" + "=" * 90)
print(f"{'job_id':<14}{'estado':<10}{'total (s)':>10}   flujo")
print("=" * 90)
for t in trabajos:
    flujo = " -> ".join(t.get("resultado", {}).get("flujo", [])) or t.get("error", "")
    print(f"{t['id']:<14}{t['estado']:<10}{segundos(t):>10.1f}   {flujo}")

tiempos = sorted(segundos(t) for t in trabajos)
print(f"\nLatencia de punta a punta (desde que se encola hasta que termina): "
      f"p50 = {statistics.median(tiempos):.1f} s | "
      f"p95 = {statistics.quantiles(tiempos, n=20, method='inclusive')[18]:.1f} s")
print(f"Completados: {sum(t['estado'] == 'DONE' for t in trabajos)} de {len(trabajos)}")

  [d688ad1f6df4] estado: DONE
  [b63102152463] estado: DONE
  [782d8d56dae2] estado: DONE
  [2ddca761231c] estado: DONE
  [ff86a014ee67] estado: DONE

job_id        estado     total (s)   flujo
782d8d56dae2  DONE           210.6   supervisor -> investigador -> supervisor -> analista -> supervisor -> sintesis
d688ad1f6df4  DONE           213.0   supervisor -> investigador -> supervisor -> analista -> supervisor -> sintesis
b63102152463  DONE           400.3   supervisor -> investigador -> supervisor -> analista -> supervisor -> sintesis
2ddca761231c  DONE           370.3   supervisor -> investigador -> supervisor -> analista -> supervisor -> sintesis
ff86a014ee67  DONE           447.0   supervisor -> investigador -> supervisor -> analista -> supervisor -> sintesis

Latencia de punta a punta (desde que se encola hasta que termina): p50 = 370.3 s | p95 = 437.7 s
Completados: 5 de 5


In [40]:
import httpx
from google.colab import output

try:
    print("Phoenix responde ->", httpx.get("http://localhost:6006", timeout=10).status_code)
except Exception as e:
    print("Phoenix NO responde:", type(e).__name__)

output.serve_kernel_port_as_window(6006, anchor_text="Abrir Phoenix")

Phoenix responde -> 200
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [41]:
import httpx

PHOENIX = "http://localhost:6006"
PROYECTO = "orquestador-api"
TRABAJOS_CARGA = {f"trabajo {j}" for j in JOBS_CARGA}

# 1. Leer todos los spans del proyecto
spans, cursor = [], None
while True:
    params = {"limit": 1000, **({"cursor": cursor} if cursor else {})}
    r = httpx.get(f"{PHOENIX}/v1/projects/{PROYECTO}/spans", params=params, timeout=60)
    r.raise_for_status()
    datos = r.json()
    spans += datos["data"]
    cursor = datos.get("next_cursor")
    if not cursor:
        break

# 2. Separar las trazas de la carga del resto (cada traza tiene un span raíz)
raices = [s for s in spans if not s.get("parent_id")]
borrar = {s["context"]["trace_id"] for s in raices if s["name"] not in TRABAJOS_CARGA}
conservar = [s["name"] for s in raices if s["name"] in TRABAJOS_CARGA]
print(f"Trazas totales: {len(raices)} | se conservan: {len(conservar)} | se borran: {len(borrar)}")
print("Se conservan:", sorted(conservar))

# 3. Borrar
estados = {}
for trace_id in borrar:
    codigo = httpx.delete(f"{PHOENIX}/v1/traces/{trace_id}", timeout=30).status_code
    estados[codigo] = estados.get(codigo, 0) + 1
print("Resultado de los borrados (código HTTP: cantidad):", estados)

Trazas totales: 476 | se conservan: 5 | se borran: 471
Se conservan: ['trabajo 2ddca761231c', 'trabajo 782d8d56dae2', 'trabajo b63102152463', 'trabajo d688ad1f6df4', 'trabajo ff86a014ee67']
Resultado de los borrados (código HTTP: cantidad): {204: 471}


## 19. Archivos del repositorio

In [42]:
%%writefile requirements.txt
fastapi
uvicorn[standard]
httpx
redis
langgraph
langgraph-checkpoint-redis
langchain-core
langchain-google-genai
arize-phoenix-otel
openinference-instrumentation-langchain
python-dotenv

Writing requirements.txt


In [43]:
%%writefile .env.example
GOOGLE_API_KEY=tu_clave_de_gemini
REDIS_URL=redis://localhost:6379
PHOENIX_COLLECTOR_ENDPOINT=http://localhost:6006/v1/traces
PHOENIX_PROJECT_NAME=orquestador-api
MODELOS_SUPERVISOR=gemini-3.1-flash-lite,gemini-3.5-flash-lite,gemini-flash-lite-latest
MODELOS_ESPECIALISTAS=gemini-3.5-flash,gemini-3.1-flash-lite,gemini-3.5-flash-lite
WORKERS=2
PAUSA_ENTRE_NODOS=5

Writing .env.example


In [44]:
%%writefile .gitignore
.env
__pycache__/
*.pyc
.venv/
*.log

Writing .gitignore


In [45]:
%%writefile Dockerfile
FROM python:3.12-slim
WORKDIR /code
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app ./app
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]

Writing Dockerfile


In [46]:
%%writefile docker-compose.yml
services:
  redis:
    image: redis:8            # Redis 8 incluye los módulos JSON y Search que usa RedisSaver
    ports:
      - "6379:6379"

  phoenix:
    image: arizephoenix/phoenix:latest
    ports:
      - "6006:6006"           # interfaz web y colector de trazas

  api:
    build: .
    env_file: .env
    environment:
      REDIS_URL: redis://redis:6379
      PHOENIX_COLLECTOR_ENDPOINT: http://phoenix:6006/v1/traces
    ports:
      - "8000:8000"
    depends_on:
      - redis
      - phoenix

Writing docker-compose.yml


In [47]:
import os
os.makedirs("scripts", exist_ok=True)

In [48]:
%%writefile scripts/prueba_carga.py
"""Prueba de carga: envía 5 peticiones concurrentes a la API y mide la latencia de punta a punta."""
import asyncio
import statistics
import sys
import time

import httpx

API = sys.argv[1] if len(sys.argv) > 1 else "http://localhost:8000"
ESTADOS_FINALES = {"DONE", "FAILED", "WAITING_APPROVAL"}

CONSULTAS = [
    "Compará los Pulse X1 y los Nova Pro: puntaje promedio, opinión de los clientes y precio/calidad.",
    "Compará los Basic 200 y los Pulse X1: puntaje promedio, opinión de los clientes y cuál conviene.",
    "Compará los Nova Pro y los Basic 200: puntaje promedio y relación precio/calidad.",
    "¿Qué opinan los clientes de los Basic 200? Quiero el puntaje promedio y el análisis de sentimiento.",
    "Compará los tres modelos de auriculares (Pulse X1, Nova Pro y Basic 200) por precio/calidad.",
]


async def esperar(cliente: httpx.AsyncClient, job_id: str, maximo: int = 1800) -> dict:
    for _ in range(maximo // 3):
        try:
            trabajo = (await cliente.get(f"{API}/tasks/{job_id}")).json()
        except httpx.TransportError:
            await asyncio.sleep(1)
            continue
        if trabajo["estado"] in ESTADOS_FINALES:
            return trabajo
        await asyncio.sleep(3)
    raise TimeoutError(job_id)


async def enviar(cliente: httpx.AsyncClient, consulta: str) -> dict:
    inicio = time.perf_counter()
    r = await cliente.post(f"{API}/tasks", json={"consulta": consulta})
    post_ms = (time.perf_counter() - inicio) * 1000
    trabajo = await esperar(cliente, r.json()["job_id"])
    return {"job_id": trabajo["id"], "estado": trabajo["estado"], "post_ms": post_ms,
            "total_s": time.perf_counter() - inicio}


async def main() -> None:
    async with httpx.AsyncClient(timeout=30) as cliente:
        resultados = await asyncio.gather(*(enviar(cliente, c) for c in CONSULTAS))

    for x in resultados:
        print(f"{x['job_id']}  {x['estado']:<10} POST {x['post_ms']:6.0f} ms   total {x['total_s']:6.1f} s")
    tiempos = sorted(x["total_s"] for x in resultados)
    print(f"\np50 = {statistics.median(tiempos):.1f} s | "
          f"p95 = {statistics.quantiles(tiempos, n=20, method='inclusive')[18]:.1f} s")


if __name__ == "__main__":
    asyncio.run(main())

Writing scripts/prueba_carga.py


In [49]:
%%writefile README.md
Pre-entrega 7: API de producción y monitoreo activo

API REST con FastAPI que expone el orquestador multi-agente de la Pre-entrega 6 (Supervisor +
Investigador + Analista + Síntesis). Las tareas se encolan y se procesan en segundo plano; el estado
de los trabajos y los checkpoints de LangGraph se guardan en Redis; cada ejecución envía trazas a
Arize Phoenix; y las solicitudes críticas (reservar un producto) se pausan hasta recibir aprobación
humana.


Estructura

```
app/
  main.py            API: POST /tasks, GET /tasks/{id}, POST /tasks/{id}/approve, GET /reservas, GET /health
  worker.py          cola de trabajos en segundo plano y estado en Redis (PENDING/RUNNING/WAITING_APPROVAL/DONE/FAILED)
  graph.py           orquestador multi-agente + checkpointer de Redis (AsyncRedisSaver)
  hitl.py            nodo de aprobación humana con interrupt()
  observability.py   instrumentación con Arize Phoenix (OpenInference)
  config.py          configuración, modelos y utilidades
  state.py           estado compartido del grafo
  datos.py           base de datos simulada (productos y reseñas)
  agents/            investigador, analista, supervisor y síntesis
scripts/
  prueba_carga.py    envía 5 peticiones concurrentes y mide la latencia
screenshots/         capturas del dashboard de Phoenix
docker-compose.yml   Redis + Phoenix + API
Pre_entrega_7.ipynb  notebook de Colab donde se desarrolló y probó todo
```


Cómo levantar el entorno

Opción A: Docker Compose (Redis, Phoenix y la API juntos)

```
cp .env.example .env          (completar GOOGLE_API_KEY)
docker compose up --build
```

- API y documentación interactiva: http://localhost:8000/docs
- Phoenix: http://localhost:6006

Opción B: sin Docker (Python 3.12 o superior)

1. Iniciar Redis 8 (incluye los módulos JSON y Search que necesita RedisSaver):

```
docker run -d -p 6379:6379 redis:8
```

2. Iniciar Phoenix:

```
pip install arize-phoenix
phoenix serve
```

3. Iniciar la API:

```
python -m venv .venv
source .venv/bin/activate        (Windows: .venv\Scripts\activate)
pip install -r requirements.txt
cp .env.example .env             (completar GOOGLE_API_KEY)
uvicorn app.main:app --port 8000
```


Uso de la API

```
# Crear una tarea: devuelve el job_id de inmediato (202 Accepted)
curl -X POST localhost:8000/tasks -H "Content-Type: application/json" \
     -d '{"consulta": "Compará los Pulse X1 y los Nova Pro por precio/calidad"}'

# Consultar el estado (polling)
curl localhost:8000/tasks/<job_id>

# Tarea crítica: queda en WAITING_APPROVAL hasta que un humano responde
curl -X POST localhost:8000/tasks -H "Content-Type: application/json" \
     -d '{"consulta": "Compará los Pulse X1 y los Nova Pro y reservame el que convenga más"}'

curl -X POST localhost:8000/tasks/<job_id>/approve -H "Content-Type: application/json" \
     -d '{"aprobado": true, "comentario": "Hay stock", "operador": "Guido"}'

# Reservas registradas por la acción crítica
curl localhost:8000/reservas
```

Prueba de carga con 5 peticiones concurrentes:

```
python scripts/prueba_carga.py http://localhost:8000
```


Decisiones de diseño

- API no bloqueante: POST /tasks guarda el trabajo en Redis como PENDING, lo pone en una cola
  (asyncio.Queue) y responde enseguida con el job_id. Todo el código es async: Redis con
  redis.asyncio, el grafo con ainvoke y el LLM con su cliente asíncrono.
- Worker: dentro del mismo proceso corren 2 workers asíncronos (configurable con WORKERS) que toman
  trabajos de la cola. Limitar la concurrencia evita superar los límites por minuto de la API de Gemini.
- Estado en Redis: cada trabajo es un hash job:<id> con su estado, resultado o error, y vence a las
  24 horas. Si el agente falla en segundo plano, la excepción se captura y el estado pasa a FAILED
  con el error, así el cliente no queda haciendo polling para siempre.
- Checkpoints en Redis: el grafo se compila con AsyncRedisSaver y usa el job_id como thread_id.
  Eso permite pausar un trabajo, liberar el worker y reanudarlo más tarde desde el mismo punto.
- Human-in-the-loop: el nodo aprobacion detecta las solicitudes críticas (reservar, apartar, pagar)
  y llama a interrupt(). El trabajo queda en WAITING_APPROVAL; al recibir POST /tasks/{id}/approve
  se reanuda con Command(resume=...) y, si fue aprobado, registra la reserva en Redis. LangGraph
  implementa la pausa con una excepción interna (Interrupt), por eso Phoenix la muestra como evento
  de tipo exception aunque el nodo termina en OK.
- Observabilidad: OpenInference instrumenta LangChain y LangGraph y envía a Phoenix cada nodo, cada
  agente, cada herramienta y cada llamada al LLM con sus tokens. Además, el worker abre un span
  "trabajo <job_id>" que agrupa toda la ejecución de un trabajo en una sola traza.
- Modelos de respaldo: supervisor, síntesis y especialistas tienen una lista de modelos de Gemini;
  si uno falla (saturación 503, cupo 429), se usa el siguiente sin cortar el trabajo.


Prueba de carga y lectura del dashboard

Se enviaron 5 peticiones concurrentes. La API respondió a todas de inmediato y los 5 trabajos
terminaron en DONE con el flujo supervisor -> investigador -> supervisor -> analista -> supervisor
-> sintesis.

```
Métrica (5 ejecuciones)                         Valor        Fuente
Latencia p95 de ejecución del agente            212,24 s     Phoenix, Metrics (04_latencia_p95.png)
Latencia p50 de ejecución del agente            157,34 s     Phoenix, Metrics
Costo total de la corrida                       $0,11 USD    Phoenix, Cost (03_costo.png)
Costo por ejecución                             ~$0,022 USD  costo total / 5
Latencia p95 de punta a punta (con cola)        437,7 s      timestamps de Redis
```

- La latencia de Phoenix mide la ejecución del agente desde que un worker toma el trabajo. La de
  punta a punta incluye además la espera en la cola: con 2 workers, 3 de los 5 trabajos esperaron
  a que se liberara uno.
- Dónde se concentra la latencia: en los especialistas. En la traza 02_traza_detalle.png el
  supervisor tarda unos 2,7 s, mientras que el investigador tarda 1m 33s, de los cuales 44,8 s son
  un intento fallido con un modelo saturado antes de pasar al modelo de respaldo. Las pausas entre
  nodos para respetar el límite por minuto del plan gratuito también suman.
- Qué consume más tokens y costo: los especialistas, que usan gemini-3.5-flash y procesan todas las
  reseñas. Ese modelo concentra la mayor parte del costo, incluidos sus tokens de razonamiento
  (Reasoning). Las llamadas del supervisor usan entre 400 y 1.500 tokens con modelos lite, mucho
  más baratos.
- Limpieza del dashboard: el polling del cliente (GET /tasks/{id}) generaba cientos de trazas de
  3 ms que distorsionaban los percentiles (la p95 daba 0,01 s). Para medir la corrida se borraron
  de Phoenix todas las trazas que no eran de los 5 trabajos de la prueba.


Capturas (carpeta screenshots/)

- 01_trazas_trabajos.png: lista de trazas, una por trabajo.
- 02_traza_detalle.png: árbol de una ejecución (supervisor, especialistas, modelo de respaldo).
- 03_costo.png: costo de la corrida y costo por modelo.
- 04_latencia_p95.png: percentiles de latencia de la corrida, incluida la p95.
- 05_traza_hitl_pausa.png: ejecución que se detiene en el nodo aprobacion.
- 06_traza_hitl_evento.png: evento de interrupción registrado en ese nodo.
- 07_resumen_carga.png: resumen de los 5 trabajos de la prueba de carga.


Sobre el LLM

Se usó Gemini por su plan gratuito. El plan gratuito tiene cupos bajos por día y por minuto, por
eso hay pausas entre nodos, 2 workers y modelos de respaldo. Con un plan pago se pueden quitar las
pausas y subir la cantidad de workers.

Writing README.md


## 20. Descargar el repositorio

In [50]:
!rm -rf app/__pycache__ app/agents/__pycache__
!zip -r pre-entrega-7.zip app scripts requirements.txt .env.example .gitignore Dockerfile docker-compose.yml README.md

from google.colab import files
files.download("pre-entrega-7.zip")

  adding: app/ (stored 0%)
  adding: app/hitl.py (deflated 57%)
  adding: app/agents/ (stored 0%)
  adding: app/agents/supervisor.py (deflated 61%)
  adding: app/agents/analyst_agent.py (deflated 60%)
  adding: app/agents/research_agent.py (deflated 58%)
  adding: app/agents/__init__.py (deflated 4%)
  adding: app/agents/base.py (deflated 56%)
  adding: app/main.py (deflated 62%)
  adding: app/observability.py (deflated 49%)
  adding: app/worker.py (deflated 64%)
  adding: app/state.py (deflated 58%)
  adding: app/config.py (deflated 53%)
  adding: app/graph.py (deflated 63%)
  adding: app/__init__.py (stored 0%)
  adding: app/datos.py (deflated 66%)
  adding: scripts/ (stored 0%)
  adding: scripts/prueba_carga.py (deflated 54%)
  adding: requirements.txt (deflated 36%)
  adding: .env.example (deflated 38%)
  adding: .gitignore (deflated 8%)
  adding: Dockerfile (deflated 22%)
  adding: docker-compose.yml (deflated 43%)
  adding: README.md (deflated 57%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>